# Lab 5a — Matching: Did the Digital-Literacy Training Programme Work?
# المعمل 5أ — المطابقة: هل نجح برنامج تدريب المهارات الرقمية؟

**Module 5 — Quasi-Experiments: Matching, Difference-in-Differences and Instrumental Variables**
**الوحدة 5 — التجارب شبه المضبوطة: المطابقة والفروق في الفروق والمتغيرات الأداتية**

*SDA-DSC-213 · SDAIA Academy · Day 3, Hour 2 · 50 minutes*

---

## The situation

Injaz opened a **voluntary** digital-literacy training programme. 46.3% of the 20,000-user
cohort enrolled. Those who enrolled complete services **11.5 percentage points** more often
than those who did not — the same eye-catching headline you met in the Module 1 workshop.

Leadership wants to expand the programme nationally on the strength of that number.

**Your job:** find out how much of that +11.5 pp is the *programme* and how much is the
*people who chose it*. Nobody randomised anything, so you will buy identification with an
assumption (*افتراض*) instead of a coin flip — and then stress-test that assumption until
it either survives or breaks.

| | |
|---|---|
| **Objective** | Estimate the ATT of the training programme by propensity matching and IPW, with overlap, balance, refutation and Rosenbaum sensitivity |
| **Data** | `injaz_users.csv` — 20,000 rows |
| **Estimand** | **ATT** (*أثر المعالجة على المعالَجين*) — the effect *on those who enrolled*, not on everyone |
| **Ground truth** | The programme effect was **planted** at **+6.0 pp**. You will not look at it until the end. |

### Bilingual key terms (first use)

| English | العربية |
|---|---|
| Propensity score | درجة الميل |
| Common support / overlap | التداخل / الدعم المشترك |
| Standardised mean difference (SMD) | الفرق المعياري في المتوسطات |
| Unconfoundedness | عدم وجود مربِكات |
| Sensitivity analysis | تحليل الحساسية |


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `causal_utils` importable from anywhere under course_assets/
ROOT = Path.cwd()
while not (ROOT / "causal_utils").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from causal_utils import use_sdaia_style
use_sdaia_style()

RANDOM_SEED = 213          # fixed for every lab: reproducibility is graded
rng = np.random.default_rng(RANDOM_SEED)
print("Environment ready ·", ROOT)

In [ ]:
# --- lab scaffolding: self-checks that never crash the notebook ----------
def check(name, fn):
    """Run a self-check. Prints PASS/FAIL/TODO instead of raising, so the
    notebook always runs top-to-bottom even with unfinished exercises."""
    try:
        fn()
    except AssertionError as e:
        print(f"[ TODO ]  {name}: {e}")
        return False
    except Exception as e:
        print(f"[ ERROR ] {name}: {type(e).__name__}: {e}")
        return False
    print(f"[  OK  ]  {name}")
    return True


def needs(*objs, msg="complete the TODO cell above first"):
    """Guard downstream cells so an unfinished TODO skips rather than crashes."""
    if any(o is None for o in objs):
        print(f"[ SKIPPED ] {msg}")
        return False
    return True

print("Scaffolding loaded: use check(...) to self-test and needs(...) to guard.")

## Step 1 — Load the cohort and reproduce the naive number *(8 min)*

Before any method, reproduce the number the business is quoting. You cannot argue a
headline down if you cannot first reproduce it.

In [ ]:
users = pd.read_csv(DATA / "injaz_users.csv")
print(f"{len(users):,} users · {users['enrolled_training'].mean():.1%} enrolled "
      f"· overall completion {users['completed'].mean():.1%}")
users.head(3)

In [ ]:
# TODO: compute the NAIVE difference in completion rates, enrolled vs not enrolled.
#
#   Implement:  naive_diff = mean(completed | enrolled==1) - mean(completed | enrolled==0)
#   Store the result (a float, in PROPORTION units, not pp) in `naive_diff`.
#
naive_diff = None      # <-- replace with your calculation

def _c():
    assert naive_diff is not None, "naive_diff is still None"
    assert 0.08 < naive_diff < 0.13, f"expected roughly +10.3 pp, got {100*naive_diff:+.2f} pp"
check("naive difference reproduced", _c)

if needs(naive_diff):
    print(f"Naive difference: {100*naive_diff:+.2f} pp")

## Step 2 — State the identifying assumption *before* you estimate

The professional habit this module installs: **the assumption is the argument; the estimate
is arithmetic.** Write the sentence first, in words a non-statistician can attack.

> *Under the assumption that, conditional on age, digital literacy, prior usage, device age
> and region, enrolling in the programme is as good as random, the matched difference is the
> ATT of the programme on those who enrolled.*

Everything after this cell is arithmetic. The sentence above is what a reviewer will attack,
and it is what you must defend.

In [ ]:
IDENTIFYING_ASSUMPTION = (
    "Unconfoundedness given X: among users with the same age, digital-literacy score, "
    "prior completion rate, prior session count, device age and region, whether a user "
    "enrolled in the training programme is as good as random."
)
WHAT_WOULD_BREAK_IT = (
    "An unmeasured driver of BOTH enrolment and completion - most plausibly personal "
    "motivation, which we never observe. Rosenbaum bounds (Step 8) put a number on how "
    "strong that would have to be."
)
COVARIATES = ["age", "digital_literacy_score", "prior_completion_rate",
              "prior_sessions", "device_age_years", "region"]

print("IDENTIFYING ASSUMPTION\n  " + IDENTIFYING_ASSUMPTION)
print("\nWHAT WOULD BREAK IT\n  " + WHAT_WOULD_BREAK_IT)
print("\nAdjustment set:", COVARIATES)

## Step 3 — Propensity scores and the overlap check *(8 min)*

The **propensity score** *e(X) = P(T=1 | X)* collapses the whole covariate vector to one
number. Rosenbaum & Rubin: if treatment is ignorable given *X*, it is ignorable given *e(X)*
alone.

**The go/no-go check is overlap** (*التداخل*). Where treated and control propensity
distributions do not overlap, there is no comparison to make — no amount of modelling
invents a control unit that does not exist.

> A propensity model is **not** judged by its AUC. A model that predicts treatment perfectly
> means *zero* overlap, i.e. a catastrophe. Judge it by the balance it buys.

In [ ]:
# TODO: estimate the propensity score for `enrolled_training` on COVARIATES.
#
#   Use  causal_utils.estimate_propensity(df, treatment, covariates)
#   which returns a pd.Series aligned to the dataframe index.
#   Store it in `pscore` and also attach it as a column `users["pscore"]`.
#
from causal_utils import estimate_propensity

pscore = None      # <-- replace

def _c():
    assert pscore is not None, "pscore is still None"
    assert len(pscore) == len(users), "pscore must have one value per user"
    assert 0 < pscore.min() and pscore.max() < 1, "propensities must lie strictly in (0, 1)"
check("propensity scores estimated", _c)

if needs(pscore):
    users["pscore"] = pscore
    print(f"propensity range: [{pscore.min():.3f}, {pscore.max():.3f}]")

In [ ]:
# TODO: plot the overlap. Two overlapping histograms/densities of `pscore`,
#       one for enrolled==1 and one for enrolled==0, and annotate the common-support
#       region. Use NAVY / BLUE / ORANGE from causal_utils.
from causal_utils import NAVY, BLUE, ORANGE, TEAL, SLATE

if needs(pscore):
    fig, ax = plt.subplots(figsize=(9, 4.2))
    # TODO: two ax.hist(...) calls with alpha=0.55, bins=40, density=True
    # TODO: shade the common-support region with ax.axvspan(...)
    ax.set_title("TODO: propensity overlap")
    ax.set_xlabel("Propensity score  e(X) = P(enrolled | X)")
    ax.set_ylabel("Density")
    plt.show()

## Step 4 — Nearest-neighbour matching with a caliper *(10 min)*

Pair each enrolled user with the *unenrolled* user closest in propensity, refusing any pair
further apart than the **caliper**. The ATT is then the mean within-pair difference in
completion.

`match_nearest` returns a `MatchResult` carrying the estimate, its paired standard error,
the balance table before and after, and — crucially — the matched frame itself, which
Step 8's sensitivity analysis needs.

> **Estimand drift warning.** Unmatched treated units are *dropped*. If many are dropped, you
> are no longer estimating the ATT; you are estimating the ATT *among matchable enrollees*,
> and the memo must say so.

In [ ]:
# TODO: run nearest-neighbour propensity matching.
#
#   Use  match_nearest(df, treatment, outcome, covariates, caliper=0.02,
#                      propensity=<the Series you already estimated>)
#   Store the MatchResult in `mres`.
#
from causal_utils import match_nearest

mres = None      # <-- replace

def _c():
    assert mres is not None, "mres is still None"
    assert mres.n_treated_matched / mres.n_treated_total > 0.95, \
        "fewer than 95% of treated units matched - check the caliper / overlap"
    assert 0.03 < mres.att < 0.09, f"ATT of {100*mres.att:+.2f} pp is outside the plausible band"
check("matching ran and matched almost every treated unit", _c)

if needs(mres):
    print(mres)

### Step 4b — The balance table is the receipt

Matching that does not achieve balance has not done its job. The convention: every
covariate's **|SMD| < 0.1** after matching. Look at the *worst* row, not the average.

In [ ]:
# TODO: show the post-matching balance table and assert every |SMD| < 0.1.
#       `mres.balance_before` and `mres.balance_after` are DataFrames with an
#       `abs_smd` column. Print the worst 8 rows of each.
if needs(mres):
    pass  # TODO

def _c():
    assert mres is not None, "run the matching cell first"
    assert mres.worst_smd_after < 0.10, \
        f"worst |SMD| after matching is {mres.worst_smd_after:.3f} - not balanced"
check("post-matching balance achieved (all |SMD| < 0.1)", _c)

## Step 5 — Cross-check with inverse-probability weighting *(6 min)*

Matching **discards** unmatched units. **IPW** keeps everyone and instead reweights the
sample: treated units by *1/e(X)*, controls by *1/(1−e(X))*, building a pseudo-population in
which treatment is independent of *X*.

Two estimators, two different failure modes, one dataset. If they agree, that is mild
evidence the answer is not an artefact of either. If they disagree by a lot, one of them is
being driven by a handful of extreme weights — which is why you always check
`max_weight_share`.

In [ ]:
# TODO: estimate the effect by IPW with  ipw_ate(df, treatment, outcome, covariates).
#       Store the returned dict in `ipw`. Then print the ATE, its CI, and the
#       maximum weight share (a single unit carrying > 5% of the estimate is a red flag).
from causal_utils import ipw_ate

ipw = None      # <-- replace

def _c():
    assert ipw is not None, "ipw is still None"
    assert ipw["max_weight_share"] < 0.05, "one unit carries >5% of the estimate"
    assert abs(ipw["ate"] - mres.att) < 0.02, "IPW and matching disagree by more than 2 pp"
check("IPW agrees with matching", _c)

if needs(ipw):
    print(ipw)

## Step 6 — identify → estimate → **refute** *(12 min)*

The DoWhy workflow is *identify → estimate → refute*. You have identified (Step 2, the
assumption + adjustment set) and estimated (Steps 4–5). Now **refute**: try to break your own
result before a reviewer does.

Three refuters, each with a different logic:

| Refuter | Logic | Passes when |
|---|---|---|
| **Placebo outcome** | re-run on an outcome the treatment *cannot* cause | the "effect" is ≈ 0 |
| **Random common cause** | add an irrelevant random covariate | the estimate does not move |
| **Data subset** | re-estimate on random 70% subsets | the estimate is stable |

Our placebo outcome is **`distance_km`** — the distance from a user's home to the nearest
service centre. Training cannot move a service centre. If the model finds an "effect" on it,
the design, not the programme, is producing the number.

In [ ]:
# TODO: run the three refuters and collect the results into `refutations` (a dict).
#
#   placebo_outcome_test(df, treatment, placebo_outcome, covariates)
#   random_common_cause_test(df, treatment, outcome, covariates, n_sims=10)
#   subset_refutation(df, treatment, outcome, covariates, frac=0.7, n_sims=10)
#
#   Use placebo_outcome="distance_km" and drop it from the covariate list you pass.
from causal_utils import (placebo_outcome_test, random_common_cause_test,
                          subset_refutation)

refutations = None      # <-- replace with a dict of the three results

def _c():
    assert refutations is not None, "refutations is still None"
    assert refutations["placebo"]["passes"], "placebo outcome shows a spurious effect"
    assert refutations["random_common_cause"]["passes"], "estimate moved under a random cause"
    assert refutations["subset"]["passes"], "estimate is not stable across subsets"
check("all three refuters pass", _c)

### Optional — the same three steps in DoWhy

DoWhy and EconML are **not installed** on the SDAIA lab image, and network installs are
unreliable in the training room. The cell below shows what the identical workflow looks like
in DoWhy so you recognise it in the wild; it degrades to a message rather than an error.

The mapping is exact:

| DoWhy | `causal_utils` equivalent used above |
|---|---|
| `model.identify_effect()` | Step 2's written assumption + `InjazDAG.explain()` (Lab 6) |
| `estimate_effect(..., propensity_score_matching)` | `match_nearest` |
| `estimate_effect(..., propensity_score_weighting)` | `ipw_ate` |
| `refute_estimate(..., placebo_treatment_refuter)` | `placebo_outcome_test` |
| `refute_estimate(..., random_common_cause)` | `random_common_cause_test` |
| `refute_estimate(..., data_subset_refuter)` | `subset_refutation` |

In [ ]:
# --- OPTIONAL: DoWhy equivalent (safe to run; skips if not installed) -----
try:
    from dowhy import CausalModel

    model = CausalModel(data=users, treatment="enrolled_training",
                        outcome="completed", common_causes=COVARIATES)
    estimand = model.identify_effect()
    est = model.estimate_effect(estimand,
                                method_name="backdoor.propensity_score_weighting")
    print("DoWhy IPW ATT:", round(est.value, 4))
    for refuter in ["random_common_cause", "placebo_treatment_refuter",
                    "data_subset_refuter"]:
        print(refuter, "->", model.refute_estimate(estimand, est, method_name=refuter))

except ImportError:
    print("DoWhy is not installed on this image - this is expected and fine.\n"
          "Everything DoWhy would have done has already been done above with\n"
          "causal_utils:\n"
          "  identify -> the written assumption + adjustment set (Step 2)\n"
          "  estimate -> match_nearest / ipw_ate                  (Steps 4-5)\n"
          "  refute   -> placebo / random-common-cause / subset   (Step 6)\n"
          "The workflow is the point; the package is an implementation detail.")

## Step 7 — Rosenbaum bounds: what matching *cannot* do *(10 min)*

Every refuter above is blind to the thing most likely to be wrong: an **unobserved**
confounder. Matching offers *zero* protection against it — unlike randomisation.

**Rosenbaum bounds** put a number on the exposure. Γ (gamma) is the odds ratio by which a
hidden variable could change a user's odds of enrolling. Γ = 1 is our assumption. The
question is: **how large must Γ be before the conclusion flips?**

Read the output as one sentence for the memo:

> *"The conclusion survives an unmeasured confounder that changes the odds of enrolling by up
> to Γ\*. A confounder stronger than that would overturn it."*

In [ ]:
# TODO: run  rosenbaum_bounds(mres.matched, outcome="completed")
#       and find Gamma-star: the LARGEST gamma at which `still_significant_at_05`
#       is still True. Store the table in `rb` and the value in `gamma_star`.
from causal_utils import rosenbaum_bounds

rb = None            # <-- replace
gamma_star = None    # <-- replace

def _c():
    assert rb is not None and gamma_star is not None, "rb / gamma_star still None"
    assert 1.0 <= gamma_star <= 3.0, "gamma_star should be within the tested grid"
check("Rosenbaum bounds computed", _c)

if needs(rb, gamma_star):
    print(rb.to_string(index=False))
    print(f"\nGamma* = {gamma_star}")

## Step 8 — Compare against the planted ground truth

Everything so far was blind. The dataset was generated with a **known** effect. Now look.

In [ ]:
import json
GT = json.loads((DATA / "GROUND_TRUTH.json").read_text())["module5_user_cohort"]


def ground_truth_comparison():
    comparison = pd.DataFrame([
        {"method": "Naive difference", "estimate_pp": 100*naive_diff,
         "ci_pp": "-", "assumption": "none (invalid)"},
        {"method": "Propensity matching (ATT)", "estimate_pp": 100*mres.att,
         "ci_pp": f"[{100*mres.ci_low:+.2f}, {100*mres.ci_high:+.2f}]",
         "assumption": "unconfoundedness given X"},
        {"method": "IPW (stabilised)", "estimate_pp": 100*ipw["ate"],
         "ci_pp": f"[{100*ipw['ci_low']:+.2f}, {100*ipw['ci_high']:+.2f}]",
         "assumption": "unconfoundedness given X"},
        {"method": "PLANTED TRUTH", "estimate_pp": 100*GT["true_ATT_training"],
         "ci_pp": "-", "assumption": "known by construction"},
    ])
    print(comparison.to_string(index=False, float_format=lambda v: f"{v:+.2f}"))
    print(f"\nSelection bias explained : {100*(naive_diff - mres.att):+.2f} pp of the "
          f"{100*naive_diff:+.2f} pp headline")
    print(f"Matching error vs truth  : {100*(mres.att - GT['true_ATT_training']):+.2f} pp")
    print(f"IPW error vs truth       : {100*(ipw['ate'] - GT['true_ATT_training']):+.2f} pp")

    def _c():
        assert abs(mres.att - GT["true_ATT_training"]) < 0.01, "matched ATT is >1 pp from truth"
        assert abs(ipw["ate"] - GT["true_ATT_training"]) < 0.01, "IPW is >1 pp from truth"
    check("both estimators recovered the planted +6.0 pp within 1 pp", _c)
    return comparison


if needs(naive_diff, mres, ipw):
    comparison = ground_truth_comparison()

## Step 9 — Write the finding

One paragraph a director can act on. Effect, interval, assumption, fragility, decision.

In [ ]:
def render_finding():
    return f"""
FINDING - Digital-literacy training programme
---------------------------------------------
The programme raised service completion among those who took it by
{100*mres.att:+.2f} percentage points (95% CI {100*mres.ci_low:+.2f} to
{100*mres.ci_high:+.2f}). The raw +{100*naive_diff:.1f} pp gap in the dashboard
overstates the programme by roughly {100*(naive_diff - mres.att):.1f} points,
because more digitally-confident users were far likelier to enrol in the first
place ({mres.balance_before['abs_smd'].max():.2f} standardised units apart on
digital literacy before matching, {mres.worst_smd_after:.2f} after).

ESTIMAND   ATT - the effect on those who enrolled, not on everyone.
ASSUMPTION {IDENTIFYING_ASSUMPTION}
ROBUSTNESS Placebo outcome null; estimate stable under a random common cause and
           across 70% subsets; matching and IPW agree to 0.01 pp.
FRAGILITY  Rosenbaum Gamma* = {gamma_star:.2f}. An unmeasured driver of enrolment
           of that strength - motivation is the obvious candidate - would
           overturn the conclusion. We did not measure motivation.
DECISION   Expand as a MONITORED pilot with randomised invitations, not as an
           unconditional national rollout. A randomised invitation next quarter
           would settle in six weeks what this analysis cannot.
"""


if needs(mres, naive_diff, gamma_star):
    FINDING = render_finding()
    print(FINDING)

## What you now own

A reusable **matching evaluation kit**. Copy this block into the capstone.

In [ ]:
def build_artefact():
    return {
        "notebook": "lab5a - propensity matching evaluation",
        "reusable_pipeline": [
            "1. reproduce the naive number (you must be able to argue it down)",
            "2. WRITE the identifying assumption and the adjustment set, before estimating",
            "3. estimate_propensity(...)  -> overlap plot -> go / no-go",
            "4. match_nearest(...)        -> ATT + balance table + love plot",
            "5. ipw_ate(...)              -> a second estimator on the same assumption",
            "6. placebo_outcome_test / random_common_cause_test / subset_refutation",
            "7. rosenbaum_bounds(...)     -> Gamma*, the fragility number",
            "8. one-paragraph FINDING: effect, CI, estimand, assumption, fragility, decision",
        ],
        "artefacts_for_the_capstone": ["love plot", "overlap plot", "refutation report",
                                       "Gamma* sentence", "FINDING paragraph"],
        "numbers_recovered": {
            "naive_pp": round(100*naive_diff, 2),
            "matched_att_pp": round(100*mres.att, 2),
            "ipw_pp": round(100*ipw["ate"], 2),
            "gamma_star": gamma_star,
            "worst_smd_after": round(mres.worst_smd_after, 3),
        },
        "rule_you_can_now_state": (
            "An observational estimate without a sensitivity analysis is an opinion. "
            "The sensitivity analysis is what makes it evidence."
        ),
    }


if needs(naive_diff, mres, ipw, gamma_star):
    ARTEFACT = build_artefact()
    for k, v in ARTEFACT.items():
        print(f"\n{k}:")
        print("  " + (("\n  ".join(v)) if isinstance(v, list) else str(v)))

## Mini exercises

### 1. Debugging — the bad control *(5 min)*

A colleague "improves" the propensity model by adding `support_calls`, arguing that "support
contact obviously matters for completion." Run the cell below, then answer: **why is this
wrong, and what did it do to the estimate?**

*(Full theory in Lab 6. Here, just observe.)*

In [ ]:
# TODO: re-run match_nearest with support_calls ADDED to the covariate list,
#       compare the ATT to the correct one, and write your diagnosis in
#       DIAGNOSIS (a string).
bad_cov = COVARIATES + ["support_calls"]

bad_att = None    # <-- replace with the ATT from the contaminated model
DIAGNOSIS = ""    # <-- replace with 2 sentences: what support_calls is, and why it is fatal

if needs(bad_att):
    print(f"correct   ATT: {100*mres.att:+.2f} pp")
    print(f"'improved' ATT: {100*bad_att:+.2f} pp")
    print(DIAGNOSIS)

### 2. Assumption articulation *(5 min)*

Write, in **one sentence each**, the identifying assumption for these three Injaz
observational questions — *before* any estimation. There is no code to run; fill in the
strings and read them to your neighbour.

1. Does using the mobile app (rather than the web portal) raise completion?
2. Did the SMS reminder programme raise completion in the regions that got it?
3. Do users who contact support complete more often?

*(Question 3 is a trap. Say why.)*

In [ ]:
ASSUMPTIONS = {
    "mobile_app_vs_web": "TODO",
    "sms_reminders":     "TODO",
    "support_contact":   "TODO - and say why this one is a trap",
}
for k, v in ASSUMPTIONS.items():
    print(f"{k:22s} {v}")

### 3. Discussion — the reviewer's challenge *(5 min)*

> *"Your matched ATT survived every refuter, but you never measured motivation. Motivated
> people enrol AND complete. Your number is worthless."*

Answer in two or three sentences using your Γ\* from Step 7. A good answer concedes the
point, quantifies it, and lands on a decision.

In [ ]:
RESPONSE = """TODO - use gamma_star. Concede, quantify, decide."""
print(RESPONSE)

---

### Quiz recall (Module 5, questions 1 and 5)

1. **What does a propensity score reduce the conditioning set to?**
   → a single scalar *e(X)*; conditioning on it removes *observed* confounding.
5. **Why is a sensitivity analysis mandatory for matching?**
   → matching handles only observed confounders; sensitivity bounds the risk from
   unobserved ones.

### Next

**Lab 5b** — the reminder rollout, where a *policy turning on at a known time* replaces the
propensity score as the source of identification, and the assumption becomes **parallel
trends**.